# SV + TRGT explorer

Interactive **Plotly** view of structural variants (Kanpig-style phased BCF) and **TRGT** repeat calls for one sample. Only SV sites with a **called genotype** (non-missing `GT`) are drawn.

**Locus:** point at **indexed** whole-genome BCF/VCF if you like — `pysam` only **fetches the interval** you set (e.g. `chr6:31803187-32050925`), so you are not loading the whole genome into memory.

**Setup:** from the repo root, `pip install -r dev-requirements.txt`, then open this notebook. Short TRGT/SV/UCSC spans are widened **once** at draw time (see `MIN_SEGMENT_PX` in the last cell) so they stay visible at the default zoom; lengths are still exact in hover text, and zooming does not recompute widths (avoids `FigureWidget` / ipywidgets sync issues). For Jupyter Classic, `notebook_connected` works when `ipywidgets` is installed.

In [1]:
from __future__ import annotations

import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import pysam
import plotly.graph_objects as go
import plotly.io as pio

# Jupyter Classic / Lab: try notebook_connected first; fall back to browser if needed.
pio.renderers.default = "notebook_connected"


def find_data_dir() -> Path:
    cwd = Path.cwd()
    for p in (cwd, cwd.parent):
        if (p / "1000920.phase2.bcf").exists():
            return p
    return cwd


def parse_locus(spec: str) -> tuple[str, int, int]:
    """Parse ``chrom:start-end`` into (chrom, start, end) with 1-based inclusive coordinates.

    Commas in numbers are allowed (e.g. ``chr6:31,803,187-32,050,925``).
    Works for any contig name without spaces.
    """
    s = spec.strip().replace(",", "").replace(" ", "")
    m = re.match(r"^([^:]+):(\d+)-(\d+)$", s)
    if not m:
        raise ValueError(
            f"Bad locus {spec!r}; expected chrom:start-end (e.g. chr6:31803187-32050925)"
        )
    chrom, a, b = m.group(1), int(m.group(2)), int(m.group(3))
    if b < a:
        raise ValueError(f"Locus end < start: {spec!r}")
    return chrom, a, b


# Vertical bands (plot y) for haplotype tracks
Y = {
    "trgt_h1": 3.2,
    "trgt_h2": 2.65,
    "sv_h1": 2.05,
    "sv_h2": 1.5,
    "ucsc_simple": 0.76,
    "ucsc_rmsk": 0.52,
    "ucsc_segdup": 0.28,
}
TRACK_H = 0.24
TRACK_UCSC = 0.17

# Gap sizing for insertions (plot units, not bp)
GAP_SCALE = 0.08  # plot units per bp, capped below
GAP_MIN = 12
GAP_MAX = 800


@dataclass
class SVRow:
    pos: int
    svtype: str
    svlen: int
    gt: tuple[Optional[int], Optional[int]]
    phased: bool
    vid: str
    qual: float
    filter: str


@dataclass
class TRGTRow:
    pos: int
    end: int
    trid: str
    gt: tuple[Optional[int], Optional[int]]
    phased: bool
    al: tuple[int, ...]
    mc: tuple[int, ...]


def gt_is_called(gt: Any) -> bool:
    if gt is None:
        return False
    if not isinstance(gt, tuple):
        return False
    return all(a is not None for a in gt)


def gt_is_homozygous_reference(gt: tuple[Optional[int], ...]) -> bool:
    """True if every allele is reference (0/0, 0|0, haploid 0)."""
    return all((a or 0) == 0 for a in gt)


def _is_symbolic_alt(alt: str) -> bool:
    return len(alt) >= 2 and alt[0] == "<" and alt[-1] == ">"


def _info_svlen_abs(rec: Any) -> Optional[int]:
    raw = rec.info.get("SVLEN")
    if isinstance(raw, tuple):
        raw = raw[0] if raw else None
    if raw is None:
        return None
    return abs(int(raw))


def _deletion_bp_symbolic(rec: Any) -> int:
    """Deleted reference length for symbolic DEL (SVLEN, END, or ref span)."""
    sl = _info_svlen_abs(rec)
    if sl is not None:
        return sl
    raw_end = rec.info.get("END")
    if isinstance(raw_end, tuple):
        raw_end = raw_end[0] if raw_end else None
    if raw_end is not None:
        return max(1, int(raw_end) - int(rec.pos))
    return max(1, int(rec.stop) - int(rec.pos))


def insertion_bp_from_ref_alt(ref: str, alt: str, rec: Any) -> int:
    """Inserted bases from REF/ALT; symbolic INS/DUP uses |SVLEN| when present."""
    if not alt:
        return 0
    if _is_symbolic_alt(alt):
        u = alt.upper()
        if "INS" in u or "DUP" in u:
            sl = _info_svlen_abs(rec)
            return sl if sl is not None else 0
        return 0
    return max(0, len(alt) - len(ref))


def deletion_bp_from_ref_alt(ref: str, alt: str, rec: Any) -> int:
    """Deleted reference bases from REF/ALT; symbolic DEL uses SVLEN/END/stop."""
    if not alt:
        return 0
    if _is_symbolic_alt(alt):
        if "DEL" in alt.upper():
            return _deletion_bp_symbolic(rec)
        return 0
    return max(0, len(ref) - len(alt))


def parse_sv_rows(path: Path, chrom: str, start: int, end: int, sample: str) -> list[SVRow]:
    """Load indel SV records from REF/ALT (and symbolic alleles + SVLEN/END)."""
    rows: list[SVRow] = []
    with pysam.VariantFile(str(path)) as vf:
        if sample not in vf.header.samples:
            raise ValueError(f"Sample {sample!r} not in {path}")
        for rec in vf.fetch(chrom, start - 1, end):
            s = rec.samples[sample]
            gt = s.get("GT")
            if not gt_is_called(gt):
                continue
            gt = tuple(gt)
            if gt_is_homozygous_reference(gt):
                continue
            ref = rec.ref or ""
            alts = rec.alts or ()
            carried = sorted({a for a in gt if a is not None and a > 0})
            if not carried:
                continue
            ins_lens: list[int] = []
            del_lens: list[int] = []
            for ai in carried:
                if ai - 1 < 0 or ai - 1 >= len(alts):
                    continue
                alt = alts[ai - 1]
                ib = insertion_bp_from_ref_alt(ref, alt, rec)
                db = deletion_bp_from_ref_alt(ref, alt, rec)
                if ib > 0:
                    ins_lens.append(ib)
                if db > 0:
                    del_lens.append(db)
            if not ins_lens and not del_lens:
                continue
            pos = int(rec.pos)
            base_vid = str(rec.id or str(rec.pos))
            qual = float(rec.qual or 0.0)
            flt = ";".join(rec.filter.keys()) if rec.filter else "PASS"
            phased = bool(s.phased)
            if del_lens:
                L = max(del_lens)
                suf = "_DEL" if ins_lens else ""
                rows.append(
                    SVRow(
                        pos=pos,
                        svtype="DEL",
                        svlen=L,
                        gt=gt,
                        phased=phased,
                        vid=f"{base_vid}{suf}",
                        qual=qual,
                        filter=flt,
                    )
                )
            if ins_lens:
                L = max(ins_lens)
                suf = "_INS" if del_lens else ""
                rows.append(
                    SVRow(
                        pos=pos,
                        svtype="INS",
                        svlen=L,
                        gt=gt,
                        phased=phased,
                        vid=f"{base_vid}{suf}",
                        qual=qual,
                        filter=flt,
                    )
                )
    return rows


def _fmt_tuple_ints(x: Any) -> tuple[int, ...]:
    if x is None:
        return ()
    if isinstance(x, tuple):
        return tuple(int(v) for v in x)
    return (int(x),)


def parse_trgt_rows(path: Path, chrom: str, start: int, end: int, sample: str) -> list[TRGTRow]:
    rows: list[TRGTRow] = []
    with pysam.VariantFile(str(path)) as vf:
        if sample not in vf.header.samples:
            raise ValueError(f"Sample {sample!r} not in {path}")
        for rec in vf.fetch(chrom, start - 1, end):
            s = rec.samples[sample]
            gt = s.get("GT")
            if not gt_is_called(gt):
                continue
            gt = tuple(gt)
            if gt_is_homozygous_reference(gt):
                continue
            raw_end = rec.info.get("END")
            if isinstance(raw_end, tuple):
                raw_end = raw_end[0] if raw_end else None
            end_i = int(raw_end) if raw_end is not None else int(rec.stop)
            rows.append(
                TRGTRow(
                    pos=int(rec.pos),
                    end=end_i,
                    trid=str(rec.info.get("TRID", rec.id or "")),
                    gt=gt,
                    phased=bool(s.phased),
                    al=_fmt_tuple_ints(s.get("AL")),
                    mc=_fmt_tuple_ints(s.get("MC")),
                )
            )
    return rows


def gap_width_for_ins_bp(ins_bp: int) -> float:
    return float(min(GAP_MAX, max(GAP_MIN, GAP_SCALE * ins_bp)))


def merge_overlapping_insertion_events(
    events: list[tuple[int, int, int, int]],
) -> list[tuple[int, int, int, int]]:
    """Merge intervals on the reference; each event is (ref_lo, ref_hi, anchor, ins_bp) inclusive."""
    if not events:
        return []
    ev = sorted(events, key=lambda t: (t[0], t[1]))
    out: list[tuple[int, int, int, int]] = []
    lo, hi, anchor, bp = ev[0]
    for lo2, hi2, a2, b2 in ev[1:]:
        if lo2 <= hi:
            hi = max(hi, hi2)
            anchor = min(anchor, a2)
            bp = max(bp, b2)
        else:
            out.append((lo, hi, anchor, bp))
            lo, hi, anchor, bp = lo2, hi2, a2, b2
    out.append((lo, hi, anchor, bp))
    return out


class GappedCoords:
    def __init__(self, win_start: int, win_end: int, gap_segments: list[tuple[int, float]]):
        self.win_start = win_start
        self.win_end = win_end
        self.gap_segments = sorted(gap_segments, key=lambda x: x[0])

    def shift_before(self, r: int) -> float:
        return sum(g for p, g in self.gap_segments if p < r)

    def ref_left(self, r: int) -> float:
        """Left edge of reference base r in plot coordinates."""
        return (r - self.win_start) + self.shift_before(r)

    def plot_span(self) -> float:
        ref_w = self.win_end - self.win_start + 1
        return ref_w + sum(g for _, g in self.gap_segments)


@dataclass
class UCSCInterval:
    """1-based inclusive coordinates on the reference assembly."""

    start: int
    end: int
    hover: str


def _ucsc_mysql_str(v: Any) -> str:
    """pymysql returns some columns (e.g. simpleRepeat.sequence) as bytes."""
    if v is None:
        return ""
    if isinstance(v, (bytes, bytearray)):
        return bytes(v).decode("ascii", errors="replace")
    return str(v)


def fetch_ucsc_repeat_tracks(
    genome: str,
    chrom: str,
    win_start: int,
    win_end: int,
) -> tuple[list[UCSCInterval], list[UCSCInterval], list[UCSCInterval], Optional[str]]:
    """Load simpleRepeat, RepeatMasker (rmsk), and segmental dups overlapping the window.

    Uses UCSC public MySQL (``genome-mysql.soe.ucsc.edu``). Coordinates in the DB are
    0-based half-open; returned intervals are 1-based inclusive to match the rest of this notebook.
    """
    import pymysql

    ws, we = win_start, win_end
    simple_rows: list[UCSCInterval] = []
    rmsk_rows: list[UCSCInterval] = []
    seg_rows: list[UCSCInterval] = []
    try:
        conn = pymysql.connect(
            host="genome-mysql.soe.ucsc.edu",
            user="genome",
            database=genome,
            connect_timeout=12,
            read_timeout=90,
        )
        try:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    SELECT chromStart, chromEnd, name, period, copyNum, sequence
                    FROM simpleRepeat
                    WHERE chrom=%s AND chromEnd > %s AND chromStart < %s
                    """,
                    (chrom, ws - 1, we),
                )
                for cs, ce, name, period, copy_num, seq in cur.fetchall():
                    s1, e1 = cs + 1, ce
                    name_s = _ucsc_mysql_str(name)
                    seq_s = _ucsc_mysql_str(seq)
                    if len(seq_s) > 48:
                        seq_s = seq_s[:48] + "…"
                    hover = (
                        f"<b>simpleRepeat</b><br>{chrom}:{s1}-{e1}<br>"
                        f"{name_s} period={period} copyNum={copy_num}"
                        + (f"<br>{seq_s}" if seq_s else "")
                    )
                    simple_rows.append(UCSCInterval(s1, e1, hover))

                cur.execute(
                    """
                    SELECT genoStart, genoEnd, repName, repClass, repFamily, strand
                    FROM rmsk
                    WHERE genoName=%s AND genoEnd > %s AND genoStart < %s
                    """,
                    (chrom, ws - 1, we),
                )
                for gs, ge, rep_name, rep_class, rep_family, strand in cur.fetchall():
                    s1, e1 = gs + 1, ge
                    hover = (
                        f"<b>RepeatMasker</b><br>{chrom}:{s1}-{e1}<br>"
                        f"{_ucsc_mysql_str(rep_name)} / {_ucsc_mysql_str(rep_class)} / "
                        f"{_ucsc_mysql_str(rep_family)}<br>strand {_ucsc_mysql_str(strand)}"
                    )
                    rmsk_rows.append(UCSCInterval(s1, e1, hover))

                cur.execute(
                    """
                    SELECT chromStart, chromEnd, name, otherChrom, otherStart, otherEnd
                    FROM genomicSuperDups
                    WHERE chrom=%s AND chromEnd > %s AND chromStart < %s
                    """,
                    (chrom, ws - 1, we),
                )
                for cs, ce, name, och, os, oe in cur.fetchall():
                    s1, e1 = cs + 1, ce
                    pair_s = _ucsc_mysql_str(name)
                    och_s = _ucsc_mysql_str(och)
                    other = (
                        f"{och_s}:{int(os) + 1}-{oe}" if och_s and os is not None and oe is not None else och_s
                    )
                    hover = (
                        f"<b>SegDup</b><br>{chrom}:{s1}-{e1}<br>"
                        f"pair: {pair_s}<br>other: {other}"
                    )
                    seg_rows.append(UCSCInterval(s1, e1, hover))
        finally:
            conn.close()
    except Exception as e:
        return [], [], [], str(e)

    return simple_rows, rmsk_rows, seg_rows, None


def allele_on_haplotype(gt: tuple[Optional[int], ...], hap: int) -> Optional[int]:
    if len(gt) <= hap:
        return None
    return gt[hap]


def sv_affects_hap(r: SVRow, hap: int) -> bool:
    """Haplotype carries a non-reference allele for this SV record."""
    a0, a1 = r.gt[0], r.gt[1] if len(r.gt) > 1 else None
    if a1 is None:
        return (a0 or 0) > 0
    ah = allele_on_haplotype(r.gt, hap)
    return ah is not None and ah > 0


def trgt_hover(r: TRGTRow, hap: int, chrom: str) -> str:
    ai = allele_on_haplotype(r.gt, hap)
    if ai is None:
        return f"{r.trid}<br>hap{hap + 1}: n/a"
    al = r.al[ai] if ai < len(r.al) else "?"
    mc = r.mc[ai] if ai < len(r.mc) else "?"
    return (
        f"<b>{r.trid}</b><br>"
        f"{chrom}:{r.pos}-{r.end} (ref)<br>"
        f"hap{hap + 1}: allele {ai} &nbsp; AL={al} &nbsp; MC={mc}<br>"
        f"GT={'|'.join(str(x) for x in r.gt) if r.phased else '/'.join(str(x) for x in r.gt)}"
    )


def trgt_ref_len_bp(r: TRGTRow) -> int:
    """Reference repeat span (bp): VCF END (inclusive) minus pysam POS (0-based)."""
    return max(1, int(r.end) - int(r.pos))


def trgt_max_expansion_bp(r: TRGTRow) -> int:
    """Max (AL − ref span) over alternate alleles in GT; 0 if no expansion."""
    ref_bp = trgt_ref_len_bp(r)
    m = 0
    for ai in r.gt:
        if ai is None or ai <= 0 or ai >= len(r.al):
            continue
        m = max(m, int(r.al[ai]) - ref_bp)
    return max(0, m)


def trgt_hap_meets_min_allele_delta(
    r: TRGTRow, hap: int, min_allele_bp: int
) -> bool:
    """True if |alt - ref| (AL vs ref span, bp) is at least min_allele_bp."""
    if min_allele_bp <= 0:
        return True
    ai = allele_on_haplotype(r.gt, hap)
    if ai is None or ai >= len(r.al):
        return False
    ref_bp = trgt_ref_len_bp(r)
    return abs(int(r.al[ai]) - ref_bp) >= min_allele_bp


def trgt_expansion_bp_for_gaps(r: TRGTRow, min_allele_bp: int) -> int:
    """Max TRGT insertion expansion (AL − ref) for gap layout; uses same hap filter as drawn bars."""
    if min_allele_bp <= 0:
        return trgt_max_expansion_bp(r)
    ref_bp = trgt_ref_len_bp(r)
    m = 0
    for hap in (0, 1):
        if not trgt_hap_meets_min_allele_delta(r, hap, min_allele_bp):
            continue
        ai = allele_on_haplotype(r.gt, hap)
        if ai is None or ai >= len(r.al):
            continue
        m = max(m, int(r.al[ai]) - ref_bp)
    return max(0, m)


def build_merged_insertion_gaps(
    sv_rows: list[SVRow],
    trgt_rows: list[TRGTRow],
    min_allele_bp: int = 0,
) -> tuple[list[tuple[int, float]], dict[int, float]]:
    """Insertion gaps from SV INS + TRGT expansions (TRGT respects min_allele_bp like TRGT traces)."""
    events: list[tuple[int, int, int, int]] = []
    ins_positions: list[int] = []
    for r in sv_rows:
        if r.svtype != "INS":
            continue
        p = r.pos
        events.append((p, p, p, r.svlen))
        ins_positions.append(p)
    for r in trgt_rows:
        ex = trgt_expansion_bp_for_gaps(r, min_allele_bp)
        if ex <= 0:
            continue
        lo, hi = int(r.pos), int(r.end)
        if hi < lo:
            lo, hi = hi, lo
        events.append((lo, hi, int(r.pos), ex))
    clusters = merge_overlapping_insertion_events(events)
    gap_segments = sorted((c[2], gap_width_for_ins_bp(c[3])) for c in clusters)
    ins_gap_plot_w: dict[int, float] = {}
    for lo, hi, anchor, ins_bp in clusters:
        w = gap_width_for_ins_bp(ins_bp)
        for p in ins_positions:
            if lo <= p <= hi:
                ins_gap_plot_w[p] = w
    return gap_segments, ins_gap_plot_w


def trgt_sites_visible_count(rows: list[TRGTRow], min_allele_bp: int) -> int:
    """Sites with at least one haplotype passing the allele-delta filter."""
    n = 0
    for r in rows:
        if any(trgt_hap_meets_min_allele_delta(r, h, min_allele_bp) for h in (0, 1)):
            n += 1
    return n


def sv_hover(r: SVRow, hap: int) -> str:
    on = sv_affects_hap(r, hap)
    return (
        f"<b>{r.vid}</b> {r.svtype} len={r.svlen}<br>"
        f"POS {r.pos} &nbsp; FILTER={r.filter}<br>"
        f"hap{hap + 1}: {'alt' if on else 'ref'}<br>"
        f"GT={'|'.join(str(x) for x in r.gt) if r.phased else '/'.join(str(x) for x in r.gt)}"
    )


def nice_reference_ticks(ws: int, we: int, max_ticks: int = 9) -> list[int]:
    """Pick readable 1-based coordinates on a 1/2/5×10ⁿ step (not window-start + fixed stride)."""
    span = we - ws + 1
    if span <= 1:
        return [ws, we]
    nt = min(max_ticks, max(2, span))
    raw = span / (nt - 1)
    exp = int(math.floor(math.log10(raw)))
    f = raw / (10**exp)
    if f <= 1.5:
        nf = 1
    elif f <= 3:
        nf = 2
    elif f <= 7:
        nf = 5
    else:
        nf = 10
    step = max(1, int(nf * 10**exp))
    t0 = math.ceil(ws / step) * step
    ticks: list[int] = []
    t = t0
    while t <= we:
        ticks.append(int(t))
        t += step
    if not ticks:
        return [ws, we]
    # If a "nice" grid tick sits just inside a window end, Plotly stacks two labels (overlap).
    merge_bp = max(5000, step // 5)
    if ticks[0] != ws:
        if ticks[0] - ws < merge_bp:
            ticks[0] = ws
        else:
            ticks.insert(0, ws)
    if ticks[-1] != we:
        if we - ticks[-1] < merge_bp:
            ticks[-1] = we
        else:
            ticks.append(we)
    return ticks


def _expand_multiseg_line(
    segments: list[tuple[float, float]],
    y_mid: float,
    hovers: list[str],
    min_span_data: float,
    *,
    anchor_right: bool = False,
) -> tuple[list[Any], list[Any], list[str]]:
    """Rebuild Scatter line arrays; widen short spans to min_span_data (plot x units).
    If anchor_right, keep xb fixed (insertion / right-anchored spans); else center the span.
    """
    lx: list[Any] = []
    ly: list[Any] = []
    lt: list[str] = []
    ms = max(0.0, float(min_span_data))
    for (xa, xb), h in zip(segments, hovers):
        w = xb - xa
        if w <= 0:
            continue
        w2 = max(w, ms) if ms > 0 else w
        if anchor_right:
            xa2 = xb - w2
            xb2 = xb
        else:
            cx = 0.5 * (xa + xb)
            xa2 = cx - 0.5 * w2
            xb2 = cx + 0.5 * w2
        lx.extend([xa2, xb2, None])
        ly.extend([y_mid, y_mid, None])
        lt.extend([h, h, ""])
    return lx, ly, lt


def _min_span_plot_units_for_window(
    gc: GappedCoords,
    win_start: int,
    win_end: int,
    min_px: float,
    *,
    layout_width_px: float = 700.0,
    margin_l: float = 52.0,
    margin_r: float = 132.0,
) -> float:
    """Plot-x span so a segment is ~min_px wide at default zoom (no live FigureWidget updates)."""
    if min_px <= 0:
        return 0.0
    x0 = float(gc.ref_left(win_start))
    x1 = float(gc.ref_left(win_end + 1))
    if x1 <= x0:
        return 0.0
    inner = max(120.0, float(layout_width_px) - margin_l - margin_r)
    return float(min_px) * (x1 - x0) / inner


def build_figure(
    sv_path: Path,
    trgt_path: Path,
    chrom: str,
    win_start: int,
    win_end: int,
    sample: Optional[str] = None,
    *,
    genome: str = "hg38",
    ucsc_tracks: bool = True,
    min_allele_bp: int = 5,
    min_segment_px: float = 6.0,
) -> go.Figure:
    with pysam.VariantFile(str(sv_path)) as vf:
        samples = list(vf.header.samples)
    if not samples:
        raise ValueError("No samples in SV VCF")
    sample = sample or samples[0]

    sv_rows = parse_sv_rows(sv_path, chrom, win_start, win_end, sample)
    trgt_rows = parse_trgt_rows(trgt_path, chrom, win_start, win_end, sample)

    gap_segs, ins_gap_plot_w = build_merged_insertion_gaps(
        sv_rows, trgt_rows, min_allele_bp
    )
    gc = GappedCoords(win_start, win_end, gap_segs)
    ms_plot = _min_span_plot_units_for_window(
        gc, win_start, win_end, min_segment_px
    )

    ucsc_err: Optional[str] = None
    n_ucsc_simple = n_ucsc_rmsk = n_ucsc_seg = 0
    simple_u: list[UCSCInterval] = []
    rmsk_u: list[UCSCInterval] = []
    seg_u: list[UCSCInterval] = []
    if ucsc_tracks:
        simple_u, rmsk_u, seg_u, ucsc_err = fetch_ucsc_repeat_tracks(
            genome, chrom, win_start, win_end
        )
        n_ucsc_simple, n_ucsc_rmsk, n_ucsc_seg = len(simple_u), len(rmsk_u), len(seg_u)

    fig = go.Figure()

    # Reference axis line (light)
    x0, x1 = gc.ref_left(win_start), gc.ref_left(win_end + 1)
    fig.add_trace(
        go.Scatter(
            x=[x0, x1],
            y=[1.0, 1.0],
            mode="lines",
            line=dict(color="#bbb", width=2),
            name="reference",
            showlegend=False,
            hoverinfo="skip",
        )
    )

    # Insertion gaps: same gapped-x interval as SV INS — [x_end-w, x_end], x_end=ref_left(a+1).
    # Right-anchored min width (ms_plot) so rects align with INS bars, not centered on them.
    _ref_gap_y_half = 0.11
    _gap_rects: list[tuple[float, float, int, float, float, float, bool]] = []
    for anchor, w_plot in gc.gap_segments:
        if w_plot <= 0:
            continue
        x_end = float(gc.ref_left(anchor + 1))
        x_start_true = float(x_end - w_plot)
        w_draw = max(w_plot, ms_plot) if ms_plot > 0 else w_plot
        lo, hi = float(x_end - w_draw), x_end
        widened = w_draw > w_plot
        _gap_rects.append((lo, hi, anchor, w_plot, x_start_true, x_end, widened))
    _gap_rects.sort(key=lambda t: t[0])
    _merged: list[dict[str, Any]] = []
    for xa, xb, anchor, w_plot, xst, xed, widened in _gap_rects:
        if not _merged:
            _merged.append(
                {"lo": xa, "hi": xb, "items": [(anchor, w_plot, xst, xed, widened)]}
            )
            continue
        cur = _merged[-1]
        if xa <= cur["hi"]:
            cur["hi"] = max(cur["hi"], xb)
            cur["items"].append((anchor, w_plot, xst, xed, widened))
        else:
            _merged.append(
                {"lo": xa, "hi": xb, "items": [(anchor, w_plot, xst, xed, widened)]}
            )
    y_lo, y_hi = 1.0 - _ref_gap_y_half, 1.0 + _ref_gap_y_half
    for blk in _merged:
        lo, hi = blk["lo"], blk["hi"]
        items = blk["items"]
        if len(items) == 1:
            anchor, w_plot, xst, xed, widened = items[0]
            ht = (
                f"Insertion gap (gapped x)<br>after ref {anchor:,}<br>"
                f"span {xst:.2f}–{xed:.2f} (w={w_plot:.2f})"
                + ("<br><i>draw min width (right edge at anchor)</i>" if widened else "")
                + "<extra></extra>"
            )
        else:
            heads = ", ".join(str(a) for a, _, _, _, _ in items[:8])
            more = f" (+{len(items) - 8} more)" if len(items) > 8 else ""
            ht = (
                f"{len(items)} overlapping gap markers (merged)<br>ref anchors: {heads}{more}<br>"
                f"<i>merged where draw spans overlap</i><extra></extra>"
            )
        fig.add_trace(
            go.Scatter(
                x=[lo, hi, hi, lo, lo],
                y=[y_lo, y_lo, y_hi, y_hi, y_lo],
                fill="toself",
                fillcolor="rgba(148, 163, 184, 0.45)",
                line=dict(color="rgba(100, 116, 139, 0.95)", width=1),
                mode="lines",
                showlegend=False,
                hovertemplate=ht,
            )
        )

    def band_y(key: str) -> tuple[float, float]:
        yc = Y[key]
        return yc - TRACK_H / 2, yc + TRACK_H / 2

    # --- TRGT: ref span bars per haplotype ---
    colors_trgt = ("#3b82f6", "#60a5fa")
    variants_legend_title = True
    for hap in (0, 1):
        key = f"trgt_h{hap + 1}"
        y_lo, y_hi = band_y(key)
        y_mid = 0.5 * (y_lo + y_hi)
        segs: list[tuple[float, float]] = []
        hovs: list[str] = []
        n_tr = 0
        for r in trgt_rows:
            if allele_on_haplotype(r.gt, hap) is None:
                continue
            if not trgt_hap_meets_min_allele_delta(r, hap, min_allele_bp):
                continue
            xa = gc.ref_left(r.pos)
            xb = gc.ref_left(r.end + 1)
            if xb <= xa:
                continue
            n_tr += 1
            segs.append((xa, xb))
            hovs.append(trgt_hover(r, hap, chrom))
        lx, ly, lt = _expand_multiseg_line(segs, y_mid, hovs, ms_plot)
        leg_trgt: dict[str, Any] = {"legendgroup": "variants", "legendrank": 1}
        if variants_legend_title:
            leg_trgt["legendgrouptitle"] = dict(text="Variants")
            variants_legend_title = False
        if segs:
            fig.add_trace(
                go.Scatter(
                    x=lx,
                    y=ly,
                    mode="lines",
                    line=dict(width=10, color=colors_trgt[hap], simplify=False),
                    name=f"TRGT H{hap + 1} ({n_tr})",
                    hoverinfo="text",
                    text=lt,
                    **leg_trgt,
                )
            )
        else:
            # No segments (e.g. filtered): still show TRGT in legend
            fig.add_trace(
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines",
                    line=dict(width=10, color=colors_trgt[hap], simplify=False),
                    name=f"TRGT H{hap + 1} ({n_tr})",
                    hoverinfo="skip",
                    showlegend=True,
                    **leg_trgt,
                )
            )

    # --- SV: DEL and INS per haplotype ---
    col_del = "#dc2626"
    col_ins = "#16a34a"
    for hap in (0, 1):
        key = f"sv_h{hap + 1}"
        y_lo, y_hi = band_y(key)
        y_mid = 0.5 * (y_lo + y_hi)

        n_sv_del = sum(
            1 for r in sv_rows if r.svtype == "DEL" and sv_affects_hap(r, hap)
        )
        segs_d: list[tuple[float, float]] = []
        hovs_d: list[str] = []
        for r in sv_rows:
            if r.svtype != "DEL" or not sv_affects_hap(r, hap):
                continue
            xa = gc.ref_left(r.pos)
            xb = gc.ref_left(r.pos + r.svlen)
            segs_d.append((xa, xb))
            hovs_d.append(sv_hover(r, hap))
        lx_d, ly_d, lt_d = _expand_multiseg_line(segs_d, y_mid, hovs_d, ms_plot)
        if segs_d:
            leg_sv: dict[str, Any] = {"legendgroup": "variants", "legendrank": 1}
            if variants_legend_title:
                leg_sv["legendgrouptitle"] = dict(text="Variants")
                variants_legend_title = False
            fig.add_trace(
                go.Scatter(
                    x=lx_d,
                    y=ly_d,
                    mode="lines",
                    line=dict(width=15, color=col_del, simplify=False),
                    name=f"SV DEL H{hap + 1} ({n_sv_del})",
                    hoverinfo="text",
                    text=lt_d,
                    **leg_sv,
                )
            )

        n_sv_ins = sum(
            1 for r in sv_rows if r.svtype == "INS" and sv_affects_hap(r, hap)
        )
        segs_i: list[tuple[float, float]] = []
        hovs_i: list[str] = []
        for r in sv_rows:
            if r.svtype != "INS" or not sv_affects_hap(r, hap):
                continue
            x_right = gc.ref_left(r.pos + 1)
            gw = float(ins_gap_plot_w.get(r.pos, gap_width_for_ins_bp(r.svlen)))
            xa = x_right - gw
            xb = x_right
            segs_i.append((xa, xb))
            hovs_i.append(sv_hover(r, hap))
        lx_i, ly_i, lt_i = _expand_multiseg_line(
            segs_i, y_mid, hovs_i, ms_plot, anchor_right=True
        )
        if segs_i:
            leg_si: dict[str, Any] = {"legendgroup": "variants", "legendrank": 1}
            if variants_legend_title:
                leg_si["legendgrouptitle"] = dict(text="Variants")
                variants_legend_title = False
            fig.add_trace(
                go.Scatter(
                    x=lx_i,
                    y=ly_i,
                    mode="lines",
                    line=dict(width=17, color=col_ins, simplify=False),
                    name=f"SV INS H{hap + 1} ({n_sv_ins})",
                    hoverinfo="text",
                    text=lt_i,
                    **leg_si,
                )
            )

    # --- UCSC annotation tracks (after variants so legend groups stay ordered) ---
    if ucsc_tracks:
        ucsc_specs = [
            ("ucsc_segdup", seg_u, "#db2777", "SegDup (UCSC)"),
            ("ucsc_rmsk", rmsk_u, "#ca8a04", "RepeatMasker"),
            ("ucsc_simple", simple_u, "#0d9488", "simpleRepeat"),
        ]
        ucsc_legend_title = True
        for ykey, rows, color, legname in ucsc_specs:
            yc = Y[ykey]
            y_lo = yc - TRACK_UCSC / 2
            y_hi = yc + TRACK_UCSC / 2
            y_mid = 0.5 * (y_lo + y_hi)
            segs_uc: list[tuple[float, float]] = []
            hovs_uc: list[str] = []
            n_uc = 0
            for u in rows:
                xa = gc.ref_left(u.start)
                xb = gc.ref_left(u.end + 1)
                if xb <= xa:
                    continue
                n_uc += 1
                segs_uc.append((xa, xb))
                hovs_uc.append(u.hover)
            lx, ly, lt = _expand_multiseg_line(segs_uc, y_mid, hovs_uc, ms_plot)
            if segs_uc:
                leg_uc: dict[str, Any] = {"legendgroup": "ucsc", "legendrank": 2}
                if ucsc_legend_title:
                    leg_uc["legendgrouptitle"] = dict(text="UCSC")
                    ucsc_legend_title = False
                fig.add_trace(
                    go.Scatter(
                        x=lx,
                        y=ly,
                        mode="lines",
                        line=dict(width=9, color=color, simplify=False),
                        name=f"{legname} ({n_uc})",
                        hoverinfo="text",
                        text=lt,
                        **leg_uc,
                    )
                )

    # X tick marks at reference coordinates (~8 ticks)
    tick_refs = nice_reference_ticks(win_start, win_end, max_ticks=9)
    tick_x = [gc.ref_left(r) for r in tick_refs]
    tick_text = [f"{r:,}" for r in tick_refs]

    n_trgt_vis = trgt_sites_visible_count(trgt_rows, min_allele_bp)
    ref_bp_span = win_end - win_start + 1
    gap_extra_x = gc.plot_span() - ref_bp_span
    status_bits = [
        f"SV: {len(sv_rows)} sites",
        f"TRGT: {n_trgt_vis} sites",
        f"INS gaps: {len(gc.gap_segments)}",
        f"gapped x: {gc.plot_span():,.0f} = ref {ref_bp_span:,} bp + {gap_extra_x:,.0f} (insertion plot units)",
        "0/0 excluded",
        (
            f"TRGT min |alt - ref|: {min_allele_bp} bp"
            if min_allele_bp > 0
            else "TRGT min |alt - ref|: off (0)"
        ),
    ]
    if min_allele_bp > 0 and len(trgt_rows) > n_trgt_vis:
        status_bits.append(
            f"{len(trgt_rows) - n_trgt_vis} TRGT sites below threshold"
        )
    status_line = " | ".join(status_bits)

    _y_lo, _y_hi = 0.05, 3.38  # must match yaxis.range (insertion gap guides)

    fig.update_layout(
        title=dict(
            text=f"<b>{sample}</b> ({chrom}:{win_start:,}-{win_end:,})",
            x=0.02,
            xanchor="left",
            y=0.99,
            yanchor="top",
            pad=dict(t=2, b=0),
        ),
        height=500,
        margin=dict(l=52, r=132, t=120, b=168),
        xaxis=dict(
            title=dict(text="Reference position", standoff=10),
            tickvals=tick_x,
            ticktext=tick_text,
            zeroline=False,
            range=[gc.ref_left(win_start) - 0.02 * gc.plot_span(), gc.ref_left(win_end + 1) + 0.02 * gc.plot_span()],
        ),
        yaxis=dict(
            range=[_y_lo, _y_hi],
            fixedrange=True,  # zoom/pan/scroll only affect x (genomic) axis
            tickmode="array",
            tickvals=[
                Y["ucsc_segdup"],
                Y["ucsc_rmsk"],
                Y["ucsc_simple"],
                1.0,
                Y["sv_h2"],
                Y["sv_h1"],
                Y["trgt_h2"],
                Y["trgt_h1"],
            ],
            ticktext=[
                "SegDup",
                "RMask",
                "simRep",
                "ref",
                "SV H2",
                "SV H1",
                "TRGT H2",
                "TRGT H1",
            ],
            title="",
        ),
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            x=1.02,
            xanchor="left",
            xref="paper",
            yref="paper",
            font=dict(size=10),
            bgcolor="rgba(255,255,255,0.94)",
            bordercolor="#e2e8f0",
            borderwidth=1,
            tracegroupgap=24,
        ),
        hoverlabel=dict(align="left"),
    )

    # Vertical guides at gapped-x boundaries of each insertion gap (ref flank coords).
    for anchor, g_plot in gc.gap_segments:
        if g_plot <= 0:
            continue
        x_end = float(gc.ref_left(anchor + 1))
        x_start = float(x_end - g_plot)
        for xv, pos_bp in ((x_start, anchor), (x_end, anchor + 1)):
            fig.add_shape(
                type="line",
                x0=xv,
                x1=xv,
                y0=_y_lo,
                y1=_y_hi,
                xref="x",
                yref="y",
                line=dict(color="rgba(71, 85, 105, 0.45)", width=1),
                layer="below",
            )
            fig.add_annotation(
                x=xv,
                xref="x",
                y=1.0,
                yref="paper",
                text=f"{chrom}:{pos_bp:,}",
                showarrow=False,
                xanchor="center",
                yanchor="bottom",
                yshift=10,
                font=dict(size=9, color="#475569"),
                textangle=-90,
            )

    # yref="paper" is the *plot domain* (0=bottom of axes box). yanchor="bottom" at y=0
    # draws text upward into the chart; use negative y + top anchor to sit in the margin
    # below ticks and the x-axis title.
    fig.add_annotation(
        x=-0.15,
        y=-0.50,
        xref="paper",
        yref="paper",
        xanchor="left",
        yanchor="top",
        showarrow=False,
        align="left",
        text=status_line,
        font=dict(size=11, color="#64748b"),
    )

    return fig


In [3]:
# --- Configure paths, locus, and sample (edit here; whole-genome files OK if indexed) ---
DATA_DIR = find_data_dir()
SV_PATH = DATA_DIR / "1000920.phase2.bcf"
TRGT_PATH = DATA_DIR / "1000920_trgt.TR_Explorer_1.01.chr6.31803187_32050925.vcf.gz"

LOCUS = "chr6:31803187-32050925"  # 1-based inclusive; only this interval is fetched
SAMPLE = None  # None → first sample in SV BCF; or e.g. "1000920"
GENOME = "hg38"  # UCSC DB name for RepeatMasker / simpleRepeat / segDup
UCSC_TRACKS = True  # needs network access to genome-mysql.soe.ucsc.edu
MIN_ALLELE_BP = 5  # TRGT: only draw haps with |alt - ref| bp >= this (AL vs ref span; 0 = off)
MIN_SEGMENT_PX = 6.0  # min on-screen width for TRGT/SV/UCSC spans at default view (baked at build time; use 0 for exact geometry)

CHROM, WIN_START, WIN_END = parse_locus(LOCUS)

assert SV_PATH.exists(), f"Missing SV BCF: {SV_PATH}"
assert TRGT_PATH.exists(), f"Missing TRGT VCF: {TRGT_PATH}"

fig = build_figure(
    SV_PATH,
    TRGT_PATH,
    CHROM,
    WIN_START,
    WIN_END,
    sample=SAMPLE,
    genome=GENOME,
    ucsc_tracks=UCSC_TRACKS,
    min_allele_bp=MIN_ALLELE_BP,
    min_segment_px=MIN_SEGMENT_PX,
)
fig.show()